In [ ]:
import cv2
import numpy as np
import pickle
import zlib
import os
from tqdm import tqdm
from sklearn.linear_model import LinearRegression
from typing import List, Tuple

class VideoCompressor:
    def __init__(self, keyframe_interval=10, quality=25):
        # Feature detection and optical flow parameters
        self.feature_params = dict(
            maxCorners=1000,
            qualityLevel=0.3,
            minDistance=7,
            blockSize=7
        )
        self.lk_params = dict(
            winSize=(15, 15),
            maxLevel=2,
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
        )
        self.keyframe_interval = keyframe_interval
        self.quality = quality  # For JPEG compression, lower means higher compression

    def simple_motion_predictor(self, flow_data):
        """
        Simple ML-based motion prediction using Linear Regression
        """
        if not flow_data:
            return None

        # Prepare data for linear regression
        X = []  # Input features
        y_x = []  # X movement prediction
        y_y = []  # Y movement prediction

        for old_points, new_points in flow_data:
            for old, new in zip(old_points, new_points):
                X.append([old[0], old[1]])  # Original coordinates
                y_x.append(new[0] - old[0])  # X movement
                y_y.append(new[1] - old[1])  # Y movement

        # Fit linear regression models
        if len(X) > 0:
            reg_x = LinearRegression().fit(X, y_x)
            reg_y = LinearRegression().fit(X, y_y)
            return (reg_x, reg_y)

        return None

    def compress_video(self, video_path: str) -> str:
        """
        Compress video with keyframe extraction and optical flow
        """
        print(f"Compressing video: {video_path}")

        # Open video capture
        cap = cv2.VideoCapture(video_path)
        original_fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        keyframes = []
        flow_data = []
        ml_motion_model = None

        # Progress bar
        pbar = tqdm(total=total_frames, desc="Processing Frames")

        # Read first frame
        ret, prev_frame = cap.read()
        if not ret:
            raise ValueError("Cannot read the first frame of the video.")

        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        frame_idx = 0

        while True:
            ret, next_frame = cap.read()
            if not ret:
                break

            next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

            # Feature tracking
            p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **self.feature_params)
            if p0 is not None:
                p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **self.lk_params)

                # Store flow data
                if st is not None:
                    good_old = p0[st.flatten() == 1]
                    good_new = p1[st.flatten() == 1]
                    flow_segment = ([point.flatten().tolist() for point in good_old],
                                    [point.flatten().tolist() for point in good_new])
                    flow_data.append(flow_segment)

            # Store keyframe
            if frame_idx % self.keyframe_interval == 0:
                ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, self.quality])
                if ret:
                    keyframes.append(compressed_keyframe.tobytes())

            prev_gray = next_gray.copy()
            prev_frame = next_frame.copy()
            frame_idx += 1
            pbar.update(1)

        # Compute ML motion predictor (minimal ML integration)
        ml_motion_model = self.simple_motion_predictor(flow_data)

        # Add the last frame as a keyframe
        ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, self.quality])
        if ret:
            keyframes.append(compressed_keyframe.tobytes())

        cap.release()
        pbar.close()

        # Compress flow data
        flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

        # Metadata
        metadata = {
            "original_fps": original_fps,
            "total_frames": total_frames,
            "keyframe_interval": self.keyframe_interval
        }

        # Save compressed data
        compressed_path = "compressed_video_data.pkl"
        with open(compressed_path, "wb") as f:
            pickle.dump({
                "keyframes": keyframes,
                "flow_data": flow_data_compressed,
                "ml_motion_model": ml_motion_model,
                "metadata": metadata
            }, f)

        print(f"Compressed video data saved to {compressed_path}")
        return compressed_path

    def reconstruct_video(self, compressed_path: str, output_path: str) -> str:
        """
        Reconstruct video from compressed data
        """
        print(f"Reconstructing video from {compressed_path}")

        # Load compressed data
        with open(compressed_path, "rb") as f:
            compressed_data = pickle.load(f)

        keyframes = compressed_data["keyframes"]
        flow_data = pickle.loads(zlib.decompress(compressed_data["flow_data"]))
        ml_motion_model = compressed_data.get("ml_motion_model")
        metadata = compressed_data["metadata"]

        # Get original video parameters
        original_fps = metadata["original_fps"]
        total_frames = metadata["total_frames"]
        keyframe_interval = metadata["keyframe_interval"]

        # Get frame size from first keyframe
        first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
        height, width = first_frame.shape[:2]

        # Initialize video writer
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        out = cv2.VideoWriter(output_path, fourcc, original_fps, (width, height))

        # Progress bar
        pbar = tqdm(total=total_frames, desc="Reconstructing Frames")

        for i in range(len(keyframes) - 1):
            current_keyframe = cv2.imdecode(np.frombuffer(keyframes[i], np.uint8), cv2.IMREAD_COLOR)
            next_keyframe = cv2.imdecode(np.frombuffer(keyframes[i + 1], np.uint8), cv2.IMREAD_COLOR)

            # Write current keyframe
            out.write(current_keyframe)
            pbar.update(1)

            # Interpolate intermediate frames
            if ml_motion_model:
                for j in range(1, keyframe_interval):
                    t = j / keyframe_interval
                    intermediate_frame = self.interpolate_frame(current_keyframe, next_keyframe, ml_motion_model, t)
                    out.write(intermediate_frame)
                    pbar.update(1)

        pbar.close()
        out.release()

        print(f"Reconstructed video saved to {output_path}")
        return output_path

    def interpolate_frame(self, frame1, frame2, model, t):
        """
        Perform linear interpolation between two frames
        """
        return cv2.addWeighted(frame1, 1 - t, frame2, t, 0)

def main():
    input_video = "/content/drive/MyDrive/ML_Project/Avengers Parts/AvengersEndgme - 3of10.mp4"
    compressor = VideoCompressor(keyframe_interval=10)

    # Compress the video
    compressed_data_path = compressor.compress_video(input_video)

    # Reconstruct the video
    output_video_path = "reconstructed_output.avi"
    compressor.reconstruct_video(compressed_data_path, output_video_path)

if __name__ == "__main__":
    main()


Compressing video: /content/drive/MyDrive/ML_Project/Avengers Parts/AvengersEndgme - 3of10.mp4


Processing Frames: 100%|█████████▉| 1560/1561 [07:12<00:00,  3.61it/s]


Compressed video data saved to compressed_video_data.pkl
Reconstructing video from compressed_video_data.pkl


Reconstructing Frames: 100%|█████████▉| 1560/1561 [01:39<00:00, 15.65it/s]

Reconstructed video saved to reconstructed_output.avi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
'''
Average MAE: 17.6582
Average PSNR: 21.3806 dB

'''
import cv2
import numpy as np
import pickle
import zlib
from tqdm import tqdm
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
import multiprocessing
import os

class VideoCompressor:
    def __init__(self, video_path, block_size=32, num_clusters=3, keyframe_quality=85):
        self.video_path = video_path
        self.block_size = block_size
        self.num_clusters = num_clusters
        self.keyframe_quality = keyframe_quality

        # Optimized feature detection parameters
        self.feature_params = dict(
            maxCorners=500,  # Reduced from 1000
            qualityLevel=0.4,  # Increased from 0.3
            minDistance=10,  # Slightly increased
            blockSize=9  # Reduced from 7
        )
        self.lk_params = dict(
            winSize=(10, 10),  # Reduced from (15, 15)
            maxLevel=1,  # Reduced from 2
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 8, 0.05)
        )

        self.keyframes = []
        self.flow_data = []
        self.motion_labels = []

    def _sample_blocks(self, image, sample_rate=0.3):
        height, width = image.shape[:2]
        blocks = []
        for y in range(0, height, self.block_size):
            for x in range(0, width, self.block_size):
                if np.random.random() < sample_rate:
                    block = image[y:min(y + self.block_size, height), x:min(x + self.block_size, width)]
                    blocks.append(block)
        return blocks

    def compress_video(self, output_path):
        cap = cv2.VideoCapture(self.video_path)
        original_fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"Loaded video: {self.video_path}")

        ret, prev_frame = cap.read()
        if not ret:
            raise ValueError("Cannot read the first frame of the video.")

        # Sample blocks for faster processing
        blocks_first = self._sample_blocks(prev_frame)
        average_colors_first = np.array([np.mean(block, axis=(0, 1)) for block in blocks_first])

        # Reduce PCA components
        pca = PCA(n_components=2)
        reduced_colors = pca.fit_transform(average_colors_first)

        kmeans = KMeans(n_clusters=self.num_clusters, random_state=42)
        kmeans.fit(reduced_colors)

        cluster_assignments_first = kmeans.labels_
        cluster_centers = kmeans.cluster_centers_

        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        frame_idx = 0

        pbar = tqdm(total=total_frames, desc="Compressing video", unit="frame")
        motion_vectors = []

        while True:
            ret, next_frame = cap.read()
            if not ret:
                break

            next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

            # Faster optical flow calculation
            p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **self.feature_params)
            if p0 is not None and len(p0) > 10:
                p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **self.lk_params)

                # Filter out poor matches
                mask = st == 1
                good_old = p0[mask]
                good_new = p1[mask]

                if len(good_old) > 0:
                    motion = good_new - good_old
                    motion_vectors.append(motion)
                    self.flow_data.append((good_old, good_new))
                    self.motion_labels.append("normal")

            # More selective keyframe selection
            if frame_idx % 15 == 0:  # Less frequent keyframe capture
                ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, self.keyframe_quality])
                if ret:
                    self.keyframes.append(compressed_keyframe.tobytes())

            prev_gray = next_gray.copy()
            prev_frame = next_frame.copy()
            frame_idx += 1
            pbar.update(1)

        pbar.close()
        cap.release()

        # Compress flow data with better compression
        flow_data_compressed = zlib.compress(pickle.dumps(self.flow_data), level=9)

        with open(output_path, "wb") as f:
            pickle.dump({
                "keyframes": self.keyframes,
                "flow_data": flow_data_compressed,
                "motion_labels": self.motion_labels,
                "fps": original_fps,
                "total_frames": total_frames,
                "cluster_assignments_first": cluster_assignments_first,
                "cluster_centers": cluster_centers
            }, f)

        print(f"Compressed video data saved at {output_path}")


class VideoDecompressor:
    def __init__(self, compressed_path):
        self.compressed_path = compressed_path

    def decompress_video(self, output_path):
        print(f"Loading compressed data from {self.compressed_path}...")
        with open(self.compressed_path, "rb") as f:
            compressed_data = pickle.load(f)

        keyframes = compressed_data["keyframes"]
        flow_data = pickle.loads(zlib.decompress(compressed_data["flow_data"]))
        fps = compressed_data["fps"]
        total_frames = compressed_data["total_frames"]

        print(f"Decompressed data: {len(keyframes)} keyframes, {len(flow_data)} motion sets.")

        # Get original video size
        first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
        height, width = first_frame.shape[:2]

        # Create video writer
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        frames_between_keyframes = total_frames // len(keyframes) - 1
        pbar = tqdm(total=len(keyframes), desc="Reconstructing video", unit="keyframe")

        for idx, keyframe_bytes in enumerate(keyframes):
            keyframe = cv2.imdecode(np.frombuffer(keyframe_bytes, np.uint8), cv2.IMREAD_COLOR)
            out.write(keyframe)

            if idx < len(flow_data):
                good_old, good_new = flow_data[idx]
                mask = np.zeros_like(keyframe)

                # Generate intermediate frames
                for i in range(1, frames_between_keyframes + 1):
                    interpolation_factor = i / (frames_between_keyframes + 1)
                    current_points = good_old + (good_new - good_old) * interpolation_factor
                    interpolated_frame = cv2.add(keyframe, mask)
                    out.write(interpolated_frame)

            pbar.update(1)

        pbar.close()
        out.release()
        print(f"Reconstructed video saved at {output_path}")


# Example Usage
if __name__ == "__main__":
    # Compression
    compressor = VideoCompressor("Trials/AvengersEndgme - 7of10.mp4")
    compressor.compress_video("Trails/compressed_video_data.pkl")

    # Decompression
    decompressor = VideoDecompressor("Trails/compressed_video_data.pkl")
    decompressor.decompress_video("Trails/reconstructed_video.avi")



# ----- Calculation For MAE and PSNR -------


import cv2
import numpy as np
import math

def calculate_mae(frame1, frame2):
    """Calculate Mean Absolute Error (MAE) between two frames"""
    return np.mean(np.abs(frame1.astype(np.float32) - frame2.astype(np.float32)))

def calculate_psnr(frame1, frame2):
    """Calculate Peak Signal-to-Noise Ratio (PSNR) between two frames"""
    mse = np.mean((frame1.astype(np.float32) - frame2.astype(np.float32)) ** 2)
    if mse == 0:
        return 100  # No error, PSNR is 100 (perfect match)
    max_pixel = 255.0
    psnr = 10 * math.log10((max_pixel ** 2) / mse)
    return psnr

# Load the original video and the reconstructed video
original_video_path = r'1min.mp4'
reconstructed_video_path = r'reconstructed_video.avi'

cap_orig = cv2.VideoCapture(original_video_path)
cap_reconstructed = cv2.VideoCapture(reconstructed_video_path)

# Check if both videos are opened correctly
if not cap_orig.isOpened() or not cap_reconstructed.isOpened():
    print("Error opening video files")
    exit()

frame_idx = 0
mae_total = 0
psnr_total = 0
frame_count = 0

while True:
    ret_orig, orig_frame = cap_orig.read()
    ret_reconstructed, reconstructed_frame = cap_reconstructed.read()

    if not ret_orig or not ret_reconstructed:
        break

    # Resize frames to match dimensions if necessary (important for MAE and PSNR calculations)
    if orig_frame.shape != reconstructed_frame.shape:
        reconstructed_frame = cv2.resize(reconstructed_frame, (orig_frame.shape[1], orig_frame.shape[0]))

    # Calculate MAE and PSNR for each frame pair
    mae = calculate_mae(orig_frame, reconstructed_frame)
    psnr = calculate_psnr(orig_frame, reconstructed_frame)

    mae_total += mae
    psnr_total += psnr
    frame_count += 1

cap_orig.release()
cap_reconstructed.release()

# Average MAE and PSNR over all frames
average_mae = mae_total / frame_count
average_psnr = psnr_total / frame_count

print(f"Average MAE: {average_mae:.4f}")
print(f"Average PSNR: {average_psnr:.4f} dB")


In [ ]:
'''
Average MAE: 13.3702
Average PSNR: 23.6348 dB

'''


import cv2
import numpy as np
import pickle
import zlib
from sklearn.cluster import KMeans

# Parameters for feature detection and optical flow
feature_params = dict(maxCorners=1000, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

def image_to_blocks(image, block_size):
    height, width = image.shape[:2]
    blocks = []
    for y in range(0, height, block_size):
        for x in range(0, width, block_size):
            block = image[y:min(y+block_size, height), x:min(x+block_size, width)]
            blocks.append(block)
    return blocks

def calculate_average_color(blocks):
    return np.array([np.mean(block, axis=(0, 1)) for block in blocks])

# Load video and get original FPS
video_path = "1min.mp4"
cap = cv2.VideoCapture(video_path)
original_fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Store keyframes and motion data
keyframes = []
flow_data = []
frame_idx = 0

ret, prev_frame = cap.read()
if not ret:
    raise ValueError("Cannot read the first frame of the video.")

# Apply KMeans clustering to first frame
block_size = 16
num_clusters = 5
blocks_first = image_to_blocks(prev_frame, block_size)
average_colors_first = calculate_average_color(blocks_first)
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
kmeans.fit(average_colors_first)
cluster_assignments_first = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# Calculate keyframe interval based on original video length
keyframe_interval = total_frames // (total_frames // 10)  # Adjust number of keyframes

while True:
    ret, next_frame = cap.read()
    if not ret:
        break

    next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

    # Feature tracking (Shi-Tomasi)
    p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)
    if p0 is not None:
        p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **lk_params)

        # Store flow data (motion vectors)
        good_old = p0[st == 1]
        good_new = p1[st == 1]
        flow_data.append((good_old, good_new))

    # Keyframe selection based on calculated interval
    if frame_idx % keyframe_interval == 0:
        ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        if ret:
            keyframes.append(compressed_keyframe.tobytes())

    # Prepare for next iteration
    prev_gray = next_gray.copy()
    prev_frame = next_frame.copy()
    frame_idx += 1

cap.release()

# Compress flow data
flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

# Save compressed data
compressed_path = "compressed_video_data2.pkl"
with open(compressed_path, "wb") as f:
    pickle.dump({
        "keyframes": keyframes,
        "flow_data": flow_data_compressed,
        "fps": original_fps,
        "total_frames": total_frames,
        "cluster_assignments_first": cluster_assignments_first,
        "cluster_centers": cluster_centers
    }, f)
print(f"Compressed video data saved at {compressed_path}")

# Reconstructing Video from Compressed Data
with open("compressed_video_data2.pkl", "rb") as f:
    compressed_data = pickle.load(f)

keyframes = compressed_data["keyframes"]
flow_data_compressed = compressed_data["flow_data"]
flow_data = pickle.loads(zlib.decompress(flow_data_compressed))
original_fps = compressed_data["fps"]
total_frames = compressed_data["total_frames"]
cluster_assignments_first = compressed_data["cluster_assignments_first"]
cluster_centers = compressed_data["cluster_centers"]

# Retrieve original video frame size
first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
height, width = first_frame.shape[:2]

# Reconstruct the video with original FPS
reconstructed_path = "reconstructed_video2.avi"
fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(reconstructed_path, fourcc, original_fps, (width, height))

# Calculate number of intermediate frames needed to match original duration
frames_between_keyframes = total_frames // len(keyframes) - 1

# Reconstruct video using keyframes and flow data
for idx, keyframe_bytes in enumerate(keyframes):
    keyframe = cv2.imdecode(np.frombuffer(keyframe_bytes, np.uint8), cv2.IMREAD_COLOR)
    out.write(keyframe)

    if idx < len(flow_data):
        good_old, good_new = flow_data[idx]
        mask = np.zeros_like(keyframe)

        # Generate intermediate frames based on optical flow
        for i in range(1, frames_between_keyframes + 1):
            interpolation_factor = i / (frames_between_keyframes + 1)

            # Interpolate points between good_old and good_new
            current_points = good_old + (good_new - good_old) * interpolation_factor

            # Create interpolated frame
            interpolated_frame = cv2.add(keyframe, mask)
            out.write(interpolated_frame)

out.release()
print(f"Reconstructed video saved at {reconstructed_path}")

In [ ]:
# ---- last edited 23/11/2024 - meet shah
'''

Average MAE: 13.3702
Average PSNR: 23.6348 dB
'''


import cv2
import numpy as np
import pickle
import zlib

# Parameters for feature detection and optical flow
feature_params = dict(maxCorners=1000, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Load video
video_path = "1min.mp4"
cap = cv2.VideoCapture(video_path)

# Store keyframes and motion data
keyframes = []
flow_data = []

frame_idx = 0
ret, prev_frame = cap.read()
if not ret:
    raise ValueError("Cannot read the first frame of the video.")

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

while True:
    ret, next_frame = cap.read()
    if not ret:
        break

    next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

    # Feature tracking (Shi-Tomasi)
    p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)
    if p0 is not None:
        p1, st, _ = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **lk_params)

        # Store flow data (motion vectors)
        good_old = p0[st == 1]
        good_new = p1[st == 1]
        flow_data.append((good_old, good_new))

    # Keyframe selection based on frame differencing (or clustering if needed)
    if frame_idx % 10 == 0:
        ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        if ret:
            keyframes.append(compressed_keyframe.tobytes())

    # Prepare for next iteration
    prev_gray = next_gray.copy()
    prev_frame = next_frame.copy()
    frame_idx += 1

cap.release()

# Compress flow data
flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

# Save compressed data
compressed_path = "compressed_video_data.pkl"
with open(compressed_path, "wb") as f:
    pickle.dump({"keyframes": keyframes, "flow_data": flow_data_compressed}, f)

print(f"Compressed video data saved at {compressed_path}")

# Reconstructing Video from Compressed Data
with open("compressed_video_data.pkl", "rb") as f:
    compressed_data = pickle.load(f)

keyframes = compressed_data["keyframes"]
flow_data_compressed = compressed_data["flow_data"]
flow_data = pickle.loads(zlib.decompress(flow_data_compressed))

# Retrieve original video frame size (height, width)
first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
height, width, _ = first_frame.shape

# Reconstruct the video
reconstructed_path = "reconstructed_video.avi"
fps = 30  # Set to match the original video's fps
fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter(reconstructed_path, fourcc, fps, (width, height))

# Reconstruct video using keyframes and flow data
for idx, keyframe_bytes in enumerate(keyframes):
    keyframe = cv2.imdecode(np.frombuffer(keyframe_bytes, np.uint8), cv2.IMREAD_COLOR)
    out.write(keyframe)

    if idx < len(flow_data):
        good_old, good_new = flow_data[idx]
        mask = np.zeros_like(keyframe)

        # Generate intermediate frames based on optical flow
        num_intermediate_frames = 9
        for i in range(1, num_intermediate_frames + 1):
            interpolated_frame = cv2.add(keyframe, mask)
            out.write(interpolated_frame)

out.release()
print(f"Reconstructed video saved at {reconstructed_path}")


ValueError: Cannot read the first frame of the video.

In [ ]:
'''

Average MAE: 9.7401
Average PSNR: 26.0363 dB

'''


import cv2
import numpy as np
import pickle
import zlib

# Parameters for feature detection and optical flow
feature_params = dict(maxCorners=1000, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

def compress_video(video_path, keyframe_interval=10):
    cap = cv2.VideoCapture(video_path)
    original_fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    keyframes = []
    flow_data = []
    frame_idx = 0

    ret, prev_frame = cap.read()
    if not ret:
        raise ValueError("Cannot read the first frame of the video.")

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    while True:
        ret, next_frame = cap.read()
        if not ret:
            break

        next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

        # Feature tracking
        p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)
        if p0 is not None:
            p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **lk_params)

            # Store flow data
            if st is not None:
                good_old = p0[st.flatten() == 1]
                good_new = p1[st.flatten() == 1]
                # Store as nested lists with flattened coordinates
                flow_data.append(([point.flatten().tolist() for point in good_old],
                                [point.flatten().tolist() for point in good_new]))

        # Store keyframe
        if frame_idx % keyframe_interval == 0:
            ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
            if ret:
                keyframes.append(compressed_keyframe.tobytes())

        prev_gray = next_gray.copy()
        prev_frame = next_frame.copy()
        frame_idx += 1

    # Add the last frame as a keyframe
    ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
    if ret:
        keyframes.append(compressed_keyframe.tobytes())

    cap.release()

    # Compress flow data
    flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

    metadata = {
        "original_fps": original_fps,
        "total_frames": total_frames,
        "keyframe_interval": keyframe_interval
    }

    # Save compressed data
    compressed_path = "compressed_video_data.pkl"
    with open(compressed_path, "wb") as f:
        pickle.dump({
            "keyframes": keyframes,
            "flow_data": flow_data_compressed,
            "metadata": metadata
        }, f)

    return compressed_path

def reconstruct_video(compressed_path, output_path):
    # Load compressed data
    with open(compressed_path, "rb") as f:
        compressed_data = pickle.load(f)

    keyframes = compressed_data["keyframes"]
    flow_data = pickle.loads(zlib.decompress(compressed_data["flow_data"]))
    metadata = compressed_data["metadata"]

    # Get original video parameters
    original_fps = metadata["original_fps"]
    total_frames = metadata["total_frames"]
    keyframe_interval = metadata["keyframe_interval"]

    # Get frame size from first keyframe
    first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
    height, width = first_frame.shape[:2]

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    out = cv2.VideoWriter(output_path, fourcc, original_fps, (width, height))

    for i in range(len(keyframes) - 1):
        current_keyframe = cv2.imdecode(np.frombuffer(keyframes[i], np.uint8), cv2.IMREAD_COLOR)
        next_keyframe = cv2.imdecode(np.frombuffer(keyframes[i + 1], np.uint8), cv2.IMREAD_COLOR)

        # Write current keyframe
        out.write(current_keyframe)

        # Generate intermediate frames
        frames_to_generate = keyframe_interval - 1
        if i == len(keyframes) - 2:  # Last segment
            remaining_frames = total_frames - (len(keyframes) - 1) * keyframe_interval
            frames_to_generate = remaining_frames - 1

        if frames_to_generate > 0 and i < len(flow_data):
            good_old = np.array(flow_data[i][0])  # Already flattened coordinates
            good_new = np.array(flow_data[i][1])  # Already flattened coordinates

            for j in range(1, frames_to_generate + 1):
                alpha = j / (frames_to_generate + 1)

                # Interpolate feature points
                if len(good_old) > 0 and len(good_new) > 0:
                    interpolated_points = good_old + alpha * (good_new - good_old)

                    # Create intermediate frame using weighted average
                    interpolated_frame = cv2.addWeighted(current_keyframe, 1 - alpha, next_keyframe, alpha, 0)

                    # Apply motion vectors
                    for old, interp in zip(good_old, interpolated_points):
                        # Convert coordinates to integers
                        old_x, old_y = int(old[0]), int(old[1])
                        new_x, new_y = int(interp[0]), int(interp[1])

                        # Draw motion vector line
                        cv2.line(interpolated_frame, (old_x, old_y), (new_x, new_y), (0, 255, 0), 1)

                out.write(interpolated_frame)

    # Write the last keyframe
    last_frame = cv2.imdecode(np.frombuffer(keyframes[-1], np.uint8), cv2.IMREAD_COLOR)
    out.write(last_frame)

    out.release()
    return output_path

# Usage
video_path = "1min.mp4"
compressed_path = compress_video(video_path)
reconstructed_path = reconstruct_video(compressed_path, "reconstructed_video.avi")
print(f"Reconstructed video saved at {reconstructed_path}")

In [ ]:
import cv2
import numpy as np
import math

def calculate_mae(frame1, frame2):
    """Calculate Mean Absolute Error (MAE) between two frames"""
    return np.mean(np.abs(frame1.astype(np.float32) - frame2.astype(np.float32)))

def calculate_psnr(frame1, frame2):
    """Calculate Peak Signal-to-Noise Ratio (PSNR) between two frames"""
    mse = np.mean((frame1.astype(np.float32) - frame2.astype(np.float32)) ** 2)
    if mse == 0:
        return 100  # No error, PSNR is 100 (perfect match)
    max_pixel = 255.0
    psnr = 10 * math.log10((max_pixel ** 2) / mse)
    return psnr

# Load the original video and the reconstructed video
original_video_path = r'C:\Users\91798\Downloads\Test_Project2\1min.mp4'
reconstructed_video_path = r'C:\Users\91798\Downloads\Test_Project2\reconstructed_video.avi'

cap_orig = cv2.VideoCapture(original_video_path)
cap_reconstructed = cv2.VideoCapture(reconstructed_video_path)

# Check if both videos are opened correctly
if not cap_orig.isOpened() or not cap_reconstructed.isOpened():
    print("Error opening video files")
    exit()

frame_idx = 0
mae_total = 0
psnr_total = 0
frame_count = 0

while True:
    ret_orig, orig_frame = cap_orig.read()
    ret_reconstructed, reconstructed_frame = cap_reconstructed.read()

    if not ret_orig or not ret_reconstructed:
        break

    # Resize frames to match dimensions if necessary (important for MAE and PSNR calculations)
    if orig_frame.shape != reconstructed_frame.shape:
        reconstructed_frame = cv2.resize(reconstructed_frame, (orig_frame.shape[1], orig_frame.shape[0]))

    # Calculate MAE and PSNR for each frame pair
    mae = calculate_mae(orig_frame, reconstructed_frame)
    psnr = calculate_psnr(orig_frame, reconstructed_frame)

    mae_total += mae
    psnr_total += psnr
    frame_count += 1

cap_orig.release()
cap_reconstructed.release()

# Average MAE and PSNR over all frames
average_mae = mae_total / frame_count
average_psnr = psnr_total / frame_count

print(f"Average MAE: {average_mae:.4f}")
print(f"Average PSNR: {average_psnr:.4f} dB")


Average MAE: 13.3702
Average PSNR: 23.6348 dB


In [ ]:
import cv2
import numpy as np
import pickle
import zlib
from sklearn.cluster import KMeans

# Parameters for feature detection and optical flow
feature_params = dict(maxCorners=1000, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

def extract_frame_features(frame):
    """Extracts edge-based features from a frame using Canny edge detection."""
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray_frame, 100, 200)
    return edges.flatten()  # Flatten to a vector for clustering

def select_best_keyframe(frames):
    """Select the best frame from a range of frames using K-means clustering."""
    features = [extract_frame_features(frame) for frame in frames]
    features = np.array(features)

    # Use KMeans to cluster frames based on features
    kmeans = KMeans(n_clusters=1, random_state=42)
    kmeans.fit(features)

    # Find the frame closest to the centroid of the cluster (the "best" frame)
    best_frame_idx = np.argmin(np.linalg.norm(features - kmeans.cluster_centers_, axis=1))
    return frames[best_frame_idx]

def compress_video_with_ml(video_path, keyframe_interval=10):
    cap = cv2.VideoCapture(video_path)
    original_fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    keyframes = []
    flow_data = []
    frame_idx = 0
    previous_frames = []

    ret, prev_frame = cap.read()
    if not ret:
        raise ValueError("Cannot read the first frame of the video.")
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    while True:
        ret, next_frame = cap.read()
        if not ret:
            break

        next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

        # Store frames within the interval (10 to 20)
        previous_frames.append(prev_frame)
        if len(previous_frames) > keyframe_interval:
            previous_frames.pop(0)

        # Select best frame within the range using ML after 10 frames
        if len(previous_frames) == keyframe_interval:
            best_keyframe = select_best_keyframe(previous_frames)
            ret, compressed_keyframe = cv2.imencode('.jpg', best_keyframe, [cv2.IMWRITE_JPEG_QUALITY, 90])
            if ret:
                keyframes.append(compressed_keyframe.tobytes())

        prev_gray = next_gray.copy()
        prev_frame = next_frame.copy()
        frame_idx += 1

    cap.release()

    # Compress flow data using pickle and zlib
    flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

    metadata = {
        "original_fps": original_fps,
        "total_frames": total_frames,
        "keyframe_interval": keyframe_interval
    }

    # Save compressed data
    compressed_path = "compressed_video_data.pkl"
    with open(compressed_path, "wb") as f:
        pickle.dump({
            "keyframes": keyframes,
            "flow_data": flow_data_compressed,
            "metadata": metadata
        }, f)

    return compressed_path

def reconstruct_video(compressed_path, output_path):
    # Load compressed data
    with open(compressed_path, "rb") as f:
        compressed_data = pickle.load(f)

    keyframes = compressed_data["keyframes"]
    flow_data = pickle.loads(zlib.decompress(compressed_data["flow_data"]))
    metadata = compressed_data["metadata"]

    # Get original video parameters
    original_fps = metadata["original_fps"]
    total_frames = metadata["total_frames"]
    keyframe_interval = metadata["keyframe_interval"]

    # Get frame size from first keyframe
    first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
    height, width = first_frame.shape[:2]

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    out = cv2.VideoWriter(output_path, fourcc, original_fps, (width, height))

    for i in range(len(keyframes) - 1):
        current_keyframe = cv2.imdecode(np.frombuffer(keyframes[i], np.uint8), cv2.IMREAD_COLOR)
        next_keyframe = cv2.imdecode(np.frombuffer(keyframes[i + 1], np.uint8), cv2.IMREAD_COLOR)

        # Write current keyframe
        out.write(current_keyframe)

        # Generate intermediate frames
        frames_to_generate = keyframe_interval - 1
        if i == len(keyframes) - 2:  # Last segment
            remaining_frames = total_frames - (len(keyframes) - 1) * keyframe_interval
            frames_to_generate = remaining_frames - 1

        if frames_to_generate > 0 and i < len(flow_data):
            good_old = np.array(flow_data[i][0])  # Already flattened coordinates
            good_new = np.array(flow_data[i][1])  # Already flattened coordinates

            for j in range(1, frames_to_generate + 1):
                alpha = j / (frames_to_generate + 1)

                # Interpolate feature points
                if len(good_old) > 0 and len(good_new) > 0:
                    interpolated_points = good_old + alpha * (good_new - good_old)

                    # Create intermediate frame using weighted average
                    interpolated_frame = cv2.addWeighted(current_keyframe, 1 - alpha, next_keyframe, alpha, 0)

                    # Apply motion vectors
                    for old, interp in zip(good_old, interpolated_points):
                        # Convert coordinates to integers
                        old_x, old_y = int(old[0]), int(old[1])
                        new_x, new_y = int(interp[0]), int(interp[1])

                        # Draw motion vector line
                        cv2.line(interpolated_frame, (old_x, old_y), (new_x, new_y), (0, 255, 0), 1)

                out.write(interpolated_frame)

    # Write the last keyframe
    last_frame = cv2.imdecode(np.frombuffer(keyframes[-1], np.uint8), cv2.IMREAD_COLOR)
    out.write(last_frame)

    out.release()
    return output_path

# Usage
video_path = "1min.mp4"
compressed_path = compress_video_with_ml(video_path)
reconstructed_path = reconstruct_video(compressed_path, "reconstructed_video.avi")
print(f"Reconstructed video saved at {reconstructed_path}")


In [ ]:
import cv2
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
import pickle
import zlib

# Parameters for feature detection and optical flow
feature_params = dict(maxCorners=1000, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

def compute_optical_flow(prev_gray, next_gray):
    """ Compute optical flow using the Lucas-Kanade method """
    p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)
    if p0 is not None:
        p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **lk_params)
        return p0, p1, st
    return None, None, None

def select_keyframes(video_path, keyframe_interval=10, n_clusters=5):
    """ Select keyframes using KMeans clustering based on optical flow features """
    cap = cv2.VideoCapture(video_path)
    frames = []
    flow_features = []

    ret, prev_frame = cap.read()
    if not ret:
        raise ValueError("Cannot read the first frame of the video.")

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    while True:
        ret, next_frame = cap.read()
        if not ret:
            break

        next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

        # Compute optical flow
        p0, p1, st = compute_optical_flow(prev_gray, next_gray)
        if p0 is not None:
            flow_vectors = p1 - p0  # Calculate flow vectors (motion)
            flow_features.append(np.mean(flow_vectors, axis=0))  # Use mean motion as feature

        frames.append(next_frame)
        prev_gray = next_gray

    cap.release()

    # Apply KMeans to cluster frames based on flow features
    flow_features = np.array(flow_features)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(flow_features)

    # Select representative keyframes from each cluster
    keyframes = []
    for i in range(n_clusters):
        cluster_indices = np.where(labels == i)[0]
        keyframe_idx = cluster_indices[0]  # Select the first frame in the cluster
        keyframes.append(frames[keyframe_idx])

    return keyframes

def train_flow_predictor(flow_data):
    """ Train a Random Forest Regressor to predict optical flow """
    # Prepare training data (use previous flow vectors as features)
    flow_data_flat = [flow.flatten() for flow in flow_data]
    X_train = np.array(flow_data_flat[:-1])  # Features (previous flow vectors)
    y_train = np.array(flow_data_flat[1:])  # Targets (next flow vectors)

    # Train the model
    model = RandomForestRegressor(n_estimators=100)
    model.fit(X_train, y_train)

    return model

def predict_flow(model, previous_flow):
    """ Predict the next optical flow using the trained model """
    return model.predict([previous_flow.flatten()]).reshape(previous_flow.shape)

def compress_video(video_path, keyframe_interval=10):
    cap = cv2.VideoCapture(video_path)
    original_fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    keyframes = []
    flow_data = []
    frame_idx = 0
    flow_features = []

    ret, prev_frame = cap.read()
    if not ret:
        raise ValueError("Cannot read the first frame of the video.")

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    while True:
        ret, next_frame = cap.read()
        if not ret:
            break

        next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

        # Compute optical flow
        p0, p1, st = compute_optical_flow(prev_gray, next_gray)
        if p0 is not None:
            flow_vectors = p1 - p0  # Calculate flow vectors (motion)
            flow_data.append(flow_vectors)

            # Collect flow features for ML
            flow_features.append(np.mean(flow_vectors, axis=0))

        prev_gray = next_gray
        frame_idx += 1

    cap.release()

    # Train flow predictor model using past flow data
    flow_model = train_flow_predictor(flow_data)

    # Apply KMeans clustering for keyframe selection
    keyframes = select_keyframes(video_path, keyframe_interval=keyframe_interval)

    # Compress flow data
    flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

    metadata = {
        "original_fps": original_fps,
        "total_frames": total_frames,
        "keyframe_interval": keyframe_interval
    }

    # Save compressed data
    compressed_path = "compressed_video_data.pkl"
    with open(compressed_path, "wb") as f:
        pickle.dump({
            "keyframes": keyframes,
            "flow_data": flow_data_compressed,
            "metadata": metadata
        }, f)

    return compressed_path

def reconstruct_video(compressed_path, output_path):
    """ Reconstruct video from compressed data """
    # Load compressed data
    with open(compressed_path, "rb") as f:
        compressed_data = pickle.load(f)

    keyframes = compressed_data["keyframes"]
    flow_data = pickle.loads(zlib.decompress(compressed_data["flow_data"]))
    metadata = compressed_data["metadata"]

    # Get original video parameters
    original_fps = metadata["original_fps"]
    total_frames = metadata["total_frames"]
    keyframe_interval = metadata["keyframe_interval"]

    # Get frame size from first keyframe
    first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
    height, width = first_frame.shape[:2]

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    out = cv2.VideoWriter(output_path, fourcc, original_fps, (width, height))

    for i in range(len(keyframes) - 1):
        current_keyframe = cv2.imdecode(np.frombuffer(keyframes[i], np.uint8), cv2.IMREAD_COLOR)
        next_keyframe = cv2.imdecode(np.frombuffer(keyframes[i + 1], np.uint8), cv2.IMREAD_COLOR)

        # Write current keyframe
        out.write(current_keyframe)

        # Generate intermediate frames using predicted flow data
        for j in range(1, keyframe_interval):
            # Predict next flow vector using the model
            predicted_flow = predict_flow(flow_model, flow_data[i])

            # Interpolate between the keyframes using the predicted flow
            interpolated_frame = cv2.remap(current_keyframe, predicted_flow[..., 0], predicted_flow[..., 1], interpolation=cv2.INTER_LINEAR)
            out.write(interpolated_frame)

    out.release()

# Example usage
video_path = "input_video.mp4"
compressed_path = compress_video(video_path)
reconstruct_video(compressed_path, "reconstructed_video.avi")


In [ ]:
import cv2
import numpy as np
import pickle
import zlib
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from tqdm import tqdm  # Import tqdm for progress bars

# Parameters for feature detection and optical flow
feature_params = dict(maxCorners=1000, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

# Function to extract optical flow magnitude as a feature for classification
def extract_flow_magnitude(flow_data):
    # Calculate flow magnitude for each point pair in flow_data
    return [np.linalg.norm(np.array(flow[1]) - np.array(flow[0]), axis=1).mean() for flow in flow_data]

def compress_video(video_path, keyframe_interval=10):
    print(f"Starting video compression for: {video_path}")
    cap = cv2.VideoCapture(video_path)
    original_fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    keyframes = []
    flow_data = []
    frame_idx = 0

    ret, prev_frame = cap.read()
    if not ret:
        raise ValueError("Cannot read the first frame of the video.")

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

    # Add tqdm progress bar for the frames being processed
    for frame_idx in tqdm(range(total_frames), desc="Compressing video frames", unit="frame"):
        ret, next_frame = cap.read()
        if not ret:
            break

        next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

        # Feature tracking
        p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **feature_params)
        if p0 is not None:
            p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **lk_params)

            # Store flow data
            if st is not None:
                good_old = p0[st.flatten() == 1]
                good_new = p1[st.flatten() == 1]
                # Store as nested lists with flattened coordinates
                flow_data.append(([point.flatten().tolist() for point in good_old],
                                  [point.flatten().tolist() for point in good_new]))

        # Store keyframe
        if frame_idx % keyframe_interval == 0:
            ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
            if ret:
                keyframes.append(compressed_keyframe.tobytes())

        prev_gray = next_gray.copy()
        prev_frame = next_frame.copy()

    # Add the last frame as a keyframe
    ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
    if ret:
        keyframes.append(compressed_keyframe.tobytes())

    cap.release()

    # Compress flow data
    flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

    metadata = {
        "original_fps": original_fps,
        "total_frames": total_frames,
        "keyframe_interval": keyframe_interval
    }

    # Save compressed data
    compressed_path = "compressed_video_data.pkl"
    with open(compressed_path, "wb") as f:
        pickle.dump({
            "keyframes": keyframes,
            "flow_data": flow_data_compressed,
            "metadata": metadata
        }, f)

    print(f"Video compression complete. Compressed data saved at {compressed_path}")
    return compressed_path

def reconstruct_video(compressed_path, output_path):
    print(f"Starting video reconstruction from: {compressed_path}")
    # Load compressed data
    with open(compressed_path, "rb") as f:
        compressed_data = pickle.load(f)

    keyframes = compressed_data["keyframes"]
    flow_data = pickle.loads(zlib.decompress(compressed_data["flow_data"]))
    metadata = compressed_data["metadata"]

    # Get original video parameters
    original_fps = metadata["original_fps"]
    total_frames = metadata["total_frames"]
    keyframe_interval = metadata["keyframe_interval"]

    # Get frame size from first keyframe
    first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
    height, width = first_frame.shape[:2]

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    out = cv2.VideoWriter(output_path, fourcc, original_fps, (width, height))

    # Extract flow magnitudes for classification
    flow_magnitudes = extract_flow_magnitude(flow_data)

    # Example labels: 0 = normal, 1 = critical (you'd need to label your dataset)
    labels = [0 if magnitude < 10 else 1 for magnitude in flow_magnitudes]  # Example threshold

    # Train a classifier (SVM)
    X_train, X_test, y_train, y_test = train_test_split(flow_magnitudes, labels, test_size=0.2)
    classifier = SVC(kernel='linear')
    classifier.fit(np.array(X_train).reshape(-1, 1), y_train)

    # Predict the class of each frame (normal or critical)
    predicted_labels = classifier.predict(np.array(flow_magnitudes).reshape(-1, 1))

    # Add tqdm progress bar for frame reconstruction
    for i in tqdm(range(len(keyframes) - 1), desc="Reconstructing video", unit="segment"):
        current_keyframe = cv2.imdecode(np.frombuffer(keyframes[i], np.uint8), cv2.IMREAD_COLOR)
        next_keyframe = cv2.imdecode(np.frombuffer(keyframes[i + 1], np.uint8), cv2.IMREAD_COLOR)

        # Write current keyframe
        out.write(current_keyframe)

        # Generate intermediate frames
        frames_to_generate = keyframe_interval - 1
        if i == len(keyframes) - 2:  # Last segment
            remaining_frames = total_frames - (len(keyframes) - 1) * keyframe_interval
            frames_to_generate = remaining_frames - 1

        if frames_to_generate > 0:
            # Generate motion vectors using predicted labels (critical = higher intensity)
            for j in range(1, frames_to_generate + 1):
                alpha = j / (frames_to_generate + 1)

                # Interpolate keyframes
                interpolated_frame = cv2.addWeighted(current_keyframe, 1 - alpha, next_keyframe, alpha, 0)

                if predicted_labels[i] == 1:  # If classified as "critical"
                    # Optionally enhance or modify the frame when it's classified as critical
                    interpolated_frame = cv2.convertScaleAbs(interpolated_frame, alpha=1.5, beta=0)  # Enhance brightness

                out.write(interpolated_frame)

    # Write the last keyframe
    last_frame = cv2.imdecode(np.frombuffer(keyframes[-1], np.uint8), cv2.IMREAD_COLOR)
    out.write(last_frame)

    out.release()
    print(f"Reconstructed video saved at {output_path}")
    return output_path

# Usage
video_path = "1min.mp4"
compressed_path = compress_video(video_path)
reconstructed_path = reconstruct_video(compressed_path, "reconstructed_video.avi")


In [ ]:
import cv2
import numpy as np
import math
import os
import pickle

class VideoQualityAssessment:
    def __init__(self, original_video_path, reconstructed_video_path, pickle_file_path):
        """Initialize video quality assessment object"""
        self.original_video_path = original_video_path
        self.reconstructed_video_path = reconstructed_video_path
        self.pickle_file_path = pickle_file_path

        # Video captures
        self.cap_orig = cv2.VideoCapture(original_video_path)
        self.cap_reconstructed = cv2.VideoCapture(reconstructed_video_path)

        # Validate video and pickle file openings
        if not self.cap_orig.isOpened() or not self.cap_reconstructed.isOpened():
            raise ValueError("Error opening video files")
        if not os.path.exists(pickle_file_path):
            raise FileNotFoundError(f"Pickle file not found: {pickle_file_path}")

        # Initialize metrics storage
        self.metrics = {
            'mae': [],
            'psnr': [],
            'euclidean_distance': []
        }

    def calculate_mae(self, frame1, frame2):
        """Calculate Mean Absolute Error between frames"""
        return np.mean(np.abs(frame1.astype(np.float32) - frame2.astype(np.float32)))

    def calculate_psnr(self, frame1, frame2):
        """Calculate Peak Signal-to-Noise Ratio between frames"""
        mse = np.mean((frame1.astype(np.float32) - frame2.astype(np.float32)) ** 2)
        if mse == 0:
            return 100
        max_pixel = 255.0
        return 10 * math.log10((max_pixel ** 2) / mse)

    def calculate_euclidean_distance(self, frame1, frame2):
        """Calculate Euclidean distance between frames"""
        return np.linalg.norm(frame1.astype(np.float32) - frame2.astype(np.float32))

    def calculate_compression_ratio(self):
        """Calculate compression ratio between original video and pickle file"""
        original_size = os.path.getsize(self.original_video_path)
        pickle_size = os.path.getsize(self.pickle_file_path)
        return original_size / pickle_size if pickle_size != 0 else 0

    def assess_video_quality(self):
        """Perform comprehensive video quality assessment"""
        frame_count = 0

        while True:
            ret_orig, orig_frame = self.cap_orig.read()
            ret_reconstructed, reconstructed_frame = self.cap_reconstructed.read()

            # Stop if either video ends
            if not ret_orig or not ret_reconstructed:
                break

            # Resize frames to match dimensions
            if orig_frame.shape != reconstructed_frame.shape:
                reconstructed_frame = cv2.resize(reconstructed_frame,
                                                 (orig_frame.shape[1], orig_frame.shape[0]))

            # Calculate metrics for this frame
            mae = self.calculate_mae(orig_frame, reconstructed_frame)
            psnr = self.calculate_psnr(orig_frame, reconstructed_frame)
            euclidean_dist = self.calculate_euclidean_distance(orig_frame, reconstructed_frame)

            # Store metrics
            self.metrics['mae'].append(mae)
            self.metrics['psnr'].append(psnr)
            self.metrics['euclidean_distance'].append(euclidean_dist)

            frame_count += 1

        # Release video captures
        self.cap_orig.release()
        self.cap_reconstructed.release()

        # Calculate compression ratio
        compression_ratio = self.calculate_compression_ratio()

        # Compute average metrics
        results = {
            'compression_ratio': compression_ratio,
            'average_mae': np.mean(self.metrics['mae']),
            'average_psnr': np.mean(self.metrics['psnr']),
            'average_euclidean_distance': np.mean(self.metrics['euclidean_distance']),
            'total_frames': frame_count,
            'original_video_size': os.path.getsize(self.original_video_path),
            'pickle_file_size': os.path.getsize(self.pickle_file_path),
            'size_reduction_percentage': (1 - os.path.getsize(self.pickle_file_path) / os.path.getsize(self.original_video_path)) * 100
        }

        return results

    def print_assessment(self):
        """Print comprehensive video quality assessment"""
        results = self.assess_video_quality()

        print("Video Quality Assessment Results:")
        print(f"Total Frames: {results['total_frames']}")
        print(f"Original Video Size: {results['original_video_size']} bytes")
        print(f"Pickle File Size: {results['pickle_file_size']} bytes")
        print(f"Compression Ratio: {results['compression_ratio']:.2f}")
        print(f"Size Reduction: {results['size_reduction_percentage']:.2f}%")
        print(f"Average MAE: {results['average_mae']:.4f}")
        print(f"Average PSNR: {results['average_psnr']:.4f} dB")
        print(f"Average Euclidean Distance: {results['average_euclidean_distance']:.4f}")

        return results

# Example usage

original_video_path = "/content/drive/MyDrive/rough/VideoCompession/Avengers Parts/AvengersEndgme - 6of10.mp4"
reconstructed_video_path = "/content/1min_reconstructed_output.avi"
pickle_file_path = "/content/1min_compressed_video_data.pkl"

try:
    # Create assessment object
    video_assessment = VideoQualityAssessment(
        original_video_path,
        reconstructed_video_path,
        pickle_file_path
    )

    # Perform and print assessment
    video_assessment.print_assessment()

except Exception as e:
    print(f"An error occurred: {e}")



In [ ]:
import cv2
import numpy as np
import pickle
import zlib
import os
import math
from tqdm import tqdm
from sklearn.linear_model import LinearRegression

class VideoCompressor:
    def __init__(self, keyframe_interval=10, quality=25):
        # Parameters for feature detection and tracking
        self.feature_params = dict(
            maxCorners=1000,  # Maximum number of corners to track
            qualityLevel=0.3,  # Minimum quality of corner points
            minDistance=7,     # Minimum distance between tracked points
            blockSize=7
        )
        # Lucas-Kanade optical flow parameters
        self.lk_params = dict(
            winSize=(15, 15),  # Search window size for tracking
            maxLevel=2,        # Pyramid levels for multiscale tracking
            criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)  # Tracking termination criteria
        )
        self.keyframe_interval = keyframe_interval  # Interval between keyframes
        self.quality = quality  # JPEG compression quality (lower means higher compression)

    def simple_motion_predictor(self, flow_data):
        """
        Simple motion prediction using Linear Regression
        Predicts movement of points based on previous frames
        """
        if not flow_data:
            return None

        # Prepare data for regression
        X, y_x, y_y = [], [], []
        for old_points, new_points in flow_data:
            for old, new in zip(old_points, new_points):
                X.append([old[0], old[1]])  # Original point coordinates
                y_x.append(new[0] - old[0])  # X-axis movement
                y_y.append(new[1] - old[1])  # Y-axis movement

        # Train linear regression models for X and Y movement
        if len(X) > 0:
            reg_x = LinearRegression().fit(X, y_x)
            reg_y = LinearRegression().fit(X, y_y)
            return (reg_x, reg_y)

        return None

    def compress_video(self, video_path):
        """
        Compress video by extracting keyframes and tracking optical flow
        """
        cap = cv2.VideoCapture(video_path)
        original_fps = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        keyframes, flow_data = [], []
        ret, prev_frame = cap.read()
        if not ret:
            raise ValueError("Cannot read the first frame of the video.")

        # Convert first frame to grayscale for feature tracking
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        frame_idx = 0

        while True:
            ret, next_frame = cap.read()
            if not ret:
                break

            # Convert next frame to grayscale
            next_gray = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)

            # Detect good features to track
            p0 = cv2.goodFeaturesToTrack(prev_gray, mask=None, **self.feature_params)

            if p0 is not None:
                # Track features using optical flow
                p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, next_gray, p0, None, **self.lk_params)

                if st is not None:
                    # Store valid tracked points
                    good_old = p0[st.flatten() == 1]
                    good_new = p1[st.flatten() == 1]
                    flow_segment = ([point.flatten().tolist() for point in good_old],
                                    [point.flatten().tolist() for point in good_new])
                    flow_data.append(flow_segment)

            # Store keyframes at specified intervals
            if frame_idx % self.keyframe_interval == 0:
                ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, self.quality])
                if ret:
                    keyframes.append(compressed_keyframe.tobytes())

            prev_gray = next_gray.copy()
            prev_frame = next_frame.copy()
            frame_idx += 1

        # Compute motion prediction model
        ml_motion_model = self.simple_motion_predictor(flow_data)

        # Add last frame as keyframe
        ret, compressed_keyframe = cv2.imencode('.jpg', prev_frame, [cv2.IMWRITE_JPEG_QUALITY, self.quality])
        if ret:
            keyframes.append(compressed_keyframe.tobytes())

        cap.release()

        # Compress flow data for storage
        flow_data_compressed = zlib.compress(pickle.dumps(flow_data))

        # Store metadata about the compression process
        metadata = {
            "original_fps": original_fps,
            "total_frames": total_frames,
            "keyframe_interval": self.keyframe_interval
        }

        # Prepare compressed data for saving
        compressed_data = {
            "keyframes": keyframes,
            "flow_data": flow_data_compressed,
            "ml_motion_model": ml_motion_model,
            "metadata": metadata
        }

        return compressed_data

    def reconstruct_video(self, compressed_data, output_path):
        """
        Reconstruct video from compressed keyframes and motion data
        """
        keyframes = compressed_data["keyframes"]
        flow_data = pickle.loads(zlib.decompress(compressed_data["flow_data"]))
        ml_motion_model = compressed_data.get("ml_motion_model")
        metadata = compressed_data["metadata"]

        # Retrieve original video parameters
        original_fps = metadata["original_fps"]
        total_frames = metadata["total_frames"]
        keyframe_interval = metadata["keyframe_interval"]

        # Get frame dimensions from first keyframe
        first_frame = cv2.imdecode(np.frombuffer(keyframes[0], np.uint8), cv2.IMREAD_COLOR)
        height, width = first_frame.shape[:2]

        # Initialize video writer
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        out = cv2.VideoWriter(output_path, fourcc, original_fps, (width, height))

        # Reconstruct video from keyframes
        for i in range(len(keyframes) - 1):
            current_keyframe = cv2.imdecode(np.frombuffer(keyframes[i], np.uint8), cv2.IMREAD_COLOR)
            next_keyframe = cv2.imdecode(np.frombuffer(keyframes[i + 1], np.uint8), cv2.IMREAD_COLOR)

            # Write current keyframe
            out.write(current_keyframe)

            # Interpolate intermediate frames if motion model exists
            if ml_motion_model:
                for j in range(1, keyframe_interval):
                    t = j / keyframe_interval
                    intermediate_frame = self.interpolate_frame(current_keyframe, next_keyframe, ml_motion_model, t)
                    out.write(intermediate_frame)

        out.release()
        return output_path

    def interpolate_frame(self, frame1, frame2, model, t):
        """
        Simple linear frame interpolation
        """
        return cv2.addWeighted(frame1, 1 - t, frame2, t, 0)

class VideoQualityAssessment:
    def __init__(self, original_video_path, reconstructed_video_path):
        """
        Initialize video quality assessment with original and reconstructed video paths
        """
        self.original_video_path = original_video_path
        self.reconstructed_video_path = reconstructed_video_path

    def calculate_mae(self, frame1, frame2):
        """
        Calculate Mean Absolute Error between two frames
        Measures average pixel-wise difference
        """
        return np.mean(np.abs(frame1.astype(np.float32) - frame2.astype(np.float32)))

    def calculate_psnr(self, frame1, frame2):
        """
        Calculate Peak Signal-to-Noise Ratio
        Higher values indicate better quality (lower distortion)
        """
        mse = np.mean((frame1.astype(np.float32) - frame2.astype(np.float32)) ** 2)
        if mse == 0:
            return 100
        max_pixel = 255.0
        return 10 * math.log10((max_pixel ** 2) / mse)

    def assess_video_quality(self):
        """
        Compare original and reconstructed videos
        Calculate MAE and PSNR for each frame
        """
        cap_orig = cv2.VideoCapture(self.original_video_path)
        cap_reconstructed = cv2.VideoCapture(self.reconstructed_video_path)

        mae_values, psnr_values = [], []

        while True:
            ret_orig, orig_frame = cap_orig.read()
            ret_reconstructed, reconstructed_frame = cap_reconstructed.read()

            # Stop when any video ends
            if not ret_orig or not ret_reconstructed:
                break

            # Resize frames to match dimensions
            if orig_frame.shape != reconstructed_frame.shape:
                reconstructed_frame = cv2.resize(reconstructed_frame, (orig_frame.shape[1], orig_frame.shape[0]))

            # Calculate quality metrics
            mae = self.calculate_mae(orig_frame, reconstructed_frame)
            psnr = self.calculate_psnr(orig_frame, reconstructed_frame)

            mae_values.append(mae)
            psnr_values.append(psnr)

        cap_orig.release()
        cap_reconstructed.release()

        # Return average metrics
        return {
            'average_mae': np.mean(mae_values),
            'average_psnr': np.mean(psnr_values)
        }

def process_video(video_file_path):
    """
    Main processing function to compress, reconstruct, and assess video quality
    """
    # Create output directory
    os.makedirs('output', exist_ok=True)

    # Define paths for compressed and reconstructed videos
    compressed_video_path = 'output/compressed_video_data.pkl'
    reconstructed_video_path = 'output/reconstructed_video.avi'

    # Initialize video compressor
    compressor = VideoCompressor()

    # Compress video and save compressed data
    print("Compressing video...")
    compressed_data = compressor.compress_video(video_file_path)
    with open(compressed_video_path, 'wb') as f:
        pickle.dump(compressed_data, f)

    # Reconstruct video from compressed data
    print("Reconstructing video...")
    compressor.reconstruct_video(compressed_data, reconstructed_video_path)

    # Assess video quality
    print("Assessing video quality...")
    quality_assessor = VideoQualityAssessment(video_file_path, reconstructed_video_path)
    metrics = quality_assessor.assess_video_quality()

    # Print quality metrics
    print("\nVideo Compression Performance Metrics:")
    print(f"Mean Absolute Error (MAE): {metrics['average_mae']:.4f}")
    print(f"Peak Signal-to-Noise Ratio (PSNR): {metrics['average_psnr']:.4f} dB")

    return metrics

# Example usage for standalone script
if __name__ == "__main__":
    input_video = "input_video.mp4"
    process_video(input_video)